# LatentSpace Agentic Training Dataset Pipeline
### Offline Semantic Middleware & Agentic Assistant Dataset Builder (Revised Architecture)
**Pipeline Version:** 2.0.0  
**Target Assistant:** LatentSpace Local Semantic Middleware  
**Core Principle:** `original_dataset.csv` contains ONLY `raw_text` and `source` (exactly 23,384 rows).  
**Noise-Inclusive:** Retains real-world speech disfluencies, noisy transcripts, and OCR lines so the model learns error handling and clarification.  
**Strict True LLM Only:** Zero fake/heuristic annotations. Unannotated rows remain pending in `original_dataset.csv`.  
**Failed Annotations Isolation:** All LLM or validation errors are logged to `output/failed_annotations.csv`.  
**Universal Tool Contract:** `dispatch_actions`  
**Provider:** OpenRouter (`qwen/qwen3.8-27b:free`, strict free-only routing)  

```text
Multiple Source Datasets
        ↓
FETCH / EXTRACT ORIGINAL CONTENT
        ↓
ORIGINAL DATASET (raw_text + source ONLY, 23,384 rows)
        ↓
TRUE LLM AGENTIC ANNOTATION (OpenRouter API)
   ├── Success ──> VALIDATION ──> SYNTHETIC DATASET & TRAINING JSONL
   └── Failure / Error ─────────> FAILED ANNOTATIONS (failed_annotations.csv)
```


## 1. Installation
Install and verify required runtime libraries for dataset loading, text processing, and schema validation.

In [ ]:
# 1. Installation verification
import sys
import os
import subprocess

# In Google Colab, clone repo if latentspace_dataset directory is missing
if 'google.colab' in sys.modules and not os.path.exists("latentspace_dataset"):
    print("Cloning repository into Google Colab environment...")
    subprocess.check_call(["git", "clone", "https://github.com/Giedel/latentspace-dataset.git", "."])

required_pkgs = ["pandas", "datasets", "requests", "tqdm", "jsonschema"]
missing_pkgs = []
for pkg in required_pkgs:
    try:
        __import__(pkg)
    except ImportError:
        missing_pkgs.append(pkg)

if missing_pkgs:
    print(f"Installing missing packages: {missing_pkgs}")
    subprocess.check_call([sys.executable, "-m", "pip", "install"] + missing_pkgs)
else:
    print("All required packages are installed.")


## 2. Imports
Import standard library modules, data science tools, and dataset utilities.

In [ ]:
# 2. Imports
import os
import sys
import json
import re
import time
import math
import hashlib
import random
import urllib.request
import requests
from datetime import datetime, timezone
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Optional, Any, Tuple
from abc import ABC, abstractmethod

import pandas as pd
import numpy as np

print("Imports completed successfully.")


## 3. Configuration
Centralized immutable configuration for the LatentSpace dataset pipeline.

In [ ]:
# 3. Pipeline Configuration
@dataclass(frozen=True)
class PipelineConfig:
    pipeline_version: str = "2.0.0"
    random_seed: int = 42

    llm_provider: str = "openrouter"
    api_base_url: str = "https://openrouter.ai/api/v1"

    annotation_model: str = "qwen/qwen3.8-27b:free"
    synthetic_model: str = "qwen/qwen3.8-27b:free"
    validation_model: str = "qwen/qwen3.8-27b:free"
    connectivity_test_model: str = "qwen/qwen3.8-27b:free"

    allow_fallbacks: bool = False

    daily_request_limit: int = 50
    stop_before_daily_limit: bool = True

    batch_size: int = 10
    max_batch_chars: int = 12000
    max_batch_tokens: int = 4096
    fallback_to_single_on_failure: bool = True
    max_retries: int = 3
    backoff_factor: float = 2.0
    request_timeout: int = 60

    # Target baseline counts (Baseline total: exactly 23,384 records)
    target_banking77: int = 7000
    target_clinc150: int = 8000
    target_hwu64: int = 5000
    target_minds14_us: int = 563
    target_minds14_ext: int = 1246   # 654 AU + 592 GB
    target_cord_v2: int = 800
    target_sroie: int = 626
    target_funsd: int = 149
    total_target: int = 23384

    dry_run_mode: bool = False
    test_mode_batch_limit: int = 5

    base_dir: str = "latentspace_dataset"
    output_dir: str = "latentspace_dataset/output"
    cache_dir: str = "latentspace_dataset/cache"
    config_dir: str = "latentspace_dataset/config"
    prompts_dir: str = "latentspace_dataset/prompts"

config = PipelineConfig()
print(f"Pipeline Config initialized. Target records: {config.total_target:,}")


## 4. Security / Environment Validation
Safely validate credentials without printing keys, and initialize the Request Ledger and Checkpoint Manager.

In [ ]:
# 4. Security, Quota Ledger, and Checkpoint Manager
api_key = os.environ.get("OPENROUTER_API_KEY", "")
try:
    from google.colab import userdata
    api_key = api_key or userdata.get("OPENROUTER_API_KEY", "")
except Exception:
    pass

if api_key:
    masked_key = api_key[:4] + "..." + api_key[-4:] if len(api_key) > 8 else "***"
    print(f"OPENROUTER_API_KEY is configured: {masked_key}")
else:
    print("OPENROUTER_API_KEY not found. Strict True LLM mode: raw dataset will be built, 0 fake annotations generated.")

class RequestLedger:
    def __init__(self, config: PipelineConfig):
        self.config = config
        self.ledger_path = os.path.join(config.cache_dir, "request_ledger.jsonl")
        os.makedirs(os.path.dirname(self.ledger_path), exist_ok=True)
        self.production_requests_used = 0
        self.cache_hits = 0
        self.retries = 0
        self._load_state()

    def _load_state(self):
        today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
        if os.path.exists(self.ledger_path):
            with open(self.ledger_path, "r", encoding="utf-8") as f:
                for line in f:
                    try:
                        record = json.loads(line)
                        if record.get("quota_date") == today and not record.get("is_test", False):
                            self.production_requests_used += 1
                        if record.get("retry_number", 0) > 0:
                            self.retries += 1
                    except Exception:
                        pass

    def record_attempt(self, attempt_data: dict):
        today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
        attempt_data["quota_date"] = today
        if not attempt_data.get("is_test", False):
            self.production_requests_used += 1
        with open(self.ledger_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(attempt_data) + "
")

    def record_cache_hit(self):
        self.cache_hits += 1

    def can_request(self) -> bool:
        if not self.config.stop_before_daily_limit:
            return True
        return self.production_requests_used < self.config.daily_request_limit

class CheckpointManager:
    @staticmethod
    def atomic_write_json(file_path: str, data: Any):
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        temp_path = f"{file_path}.tmp.{os.getpid()}.{time.time_ns()}"
        with open(temp_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)
            f.flush()
            os.fsync(f.fileno())
        os.replace(temp_path, file_path)

    @staticmethod
    def load_json(file_path: str) -> Optional[Any]:
        if os.path.exists(file_path):
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    return json.load(f)
            except Exception:
                return None
        return None

ledger = RequestLedger(config)
print(f"RequestLedger initialized. Production requests used today: {ledger.production_requests_used}/{config.daily_request_limit}")


## 5. Controlled Domain & Category Taxonomy
Controlled taxonomy and universal tool contract for agentic routing.

In [ ]:
# 5. Controlled Taxonomy & Universal Tool Schema
DOMAINS = [
    "finance", "productivity", "travel", "commerce",
    "communication", "account_and_security", "documents",
    "information", "other"
]

CATEGORIES = [
    "finance/account", "finance/transactions", "finance/payments",
    "finance/transfers", "finance/cards", "finance/expenses",
    "productivity/tasks", "productivity/reminders", "productivity/alarms",
    "productivity/notes", "productivity/calendar",
    "travel/flights", "travel/reservations", "travel/transportation",
    "commerce/products", "commerce/orders", "commerce/subscriptions", "commerce/purchases",
    "communication/messages", "communication/email", "communication/contacts",
    "account_and_security/profile", "account_and_security/authentication",
    "account_and_security/credentials", "account_and_security/security", "account_and_security/cards",
    "documents/ocr", "documents/extraction", "documents/receipts", "documents/invoices", "documents/forms",
    "information/general_information", "information/navigation", "information/calculation",
    "other/clarification", "other/out_of_scope"
]

UNIVERSAL_DISPATCH_TOOL = {
    "type": "function",
    "function": {
        "name": "dispatch_actions",
        "description": "Execute one or more structured actions sequentially or with dependencies to fulfill the user intent.",
        "parameters": {
            "type": "object",
            "properties": {
                "actions": {
                    "type": "array",
                    "description": "Ordered list of action invocations to dispatch",
                    "items": {
                        "type": "object",
                        "properties": {
                            "action_id": {
                                "type": "string",
                                "description": "Unique identifier such as a1, a2"
                            },
                            "intent_name": {
                                "type": "string",
                                "description": "The registered intent name"
                            },
                            "depends_on": {
                                "type": "array",
                                "items": {"type": "string"},
                                "description": "List of action_ids this action depends upon"
                            },
                            "entities": {
                                "type": "object",
                                "description": "Key-value entity parameters"
                            }
                        },
                        "required": ["action_id", "intent_name", "entities"]
                    }
                }
            },
            "required": ["actions"]
        }
    }
}

print(f"Taxonomy defined: {len(DOMAINS)} domains, {len(CATEGORIES)} categories.")


## 6. Intent Registry
The strict source of truth for valid LatentSpace intents.

In [ ]:
# 6. Intent Registry
class IntentRegistry:
    def __init__(self, config_path: str = "latentspace_dataset/config/intent_registry.json"):
        self.config_path = config_path
        self.intents: Dict[str, dict] = {}
        self.load()

    def load(self):
        if os.path.exists(self.config_path):
            with open(self.config_path, "r", encoding="utf-8") as f:
                data = json.load(f)
                self.intents = data.get("intents", data)
        else:
            self.intents = {}

    def is_valid_intent(self, intent_name: str) -> bool:
        return intent_name in self.intents

    def get_intent(self, intent_name: str) -> Optional[dict]:
        return self.intents.get(intent_name)

registry = IntentRegistry(os.path.join(config.config_dir, "intent_registry.json"))
print(f"Loaded IntentRegistry: {len(registry.intents)} intents.")


## 7. Dataset Adapters
Adapters extract ONLY the original source text (`raw_text`) and the source name (`source`).
Real-world noise, mistakes, and speech artifacts are fully preserved.

In [ ]:
# 7. Dataset Adapters (Extract ONLY raw_text and source)
def is_valid_raw_text(text: Optional[str]) -> bool:
    if text is None:
        return False
    if not isinstance(text, str):
        text = str(text)
    return len(text.strip()) > 0

class DatasetAdapter(ABC):
    def __init__(self, source_name: str, target_count: int, config: PipelineConfig):
        self.source_name = source_name
        self.target_count = target_count
        self.config = config

    def ensure_source_file(self, filename: str) -> Optional[str]:
        cache_path = os.path.join(self.config.cache_dir, "sources", filename)
        if os.path.exists(cache_path):
            return cache_path
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        raw_url = f"https://raw.githubusercontent.com/Giedel/latentspace-dataset/master/latentspace_dataset/cache/sources/{filename}"
        try:
            resp = requests.get(raw_url, timeout=30)
            if resp.status_code == 200 and len(resp.content) > 100:
                with open(cache_path, "wb") as f:
                    f.write(resp.content)
                return cache_path
        except Exception:
            pass
        return None

    @abstractmethod
    def fetch_records(self) -> List[dict]:
        # Returns list of dicts with keys: 'raw_text', 'source'.
        pass

class Banking77Adapter(DatasetAdapter):
    def __init__(self, config: PipelineConfig):
        super().__init__("BANKING77", config.target_banking77, config)

    def fetch_records(self) -> List[dict]:
        cache_file = self.ensure_source_file("banking77.json") or os.path.join(self.config.cache_dir, "sources", "banking77.json")
        if os.path.exists(cache_file):
            raw = json.load(open(cache_file, "r", encoding="utf-8"))
            df_raw = pd.DataFrame(raw)
        else:
            url = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/train.csv"
            try:
                df_raw = pd.read_csv(url)
            except Exception:
                from datasets import load_dataset
                df_raw = pd.DataFrame(load_dataset("PolyAI/banking77", split="train"))

        b_target = min(self.target_count, len(df_raw))
        sample = df_raw.sample(n=b_target, random_state=self.config.random_seed)

        return [
            {
                "raw_text": str(row.get("text", "")).strip(),
                "source": "BANKING77"
            }
            for _, row in sample.iterrows()
        ]

class Clinc150Adapter(DatasetAdapter):
    def __init__(self, config: PipelineConfig):
        super().__init__("CLINC150", config.target_clinc150, config)

    def fetch_records(self) -> List[dict]:
        cache_file = self.ensure_source_file("clinc150.json") or os.path.join(self.config.cache_dir, "sources", "clinc150.json")
        if os.path.exists(cache_file):
            raw = json.load(open(cache_file, "r", encoding="utf-8"))
            c_target = min(self.target_count, len(raw))
            rng = random.Random(self.config.random_seed)
            shuffled = list(raw)
            rng.shuffle(shuffled)
            selected = shuffled[:c_target]
            return [
                {
                    "raw_text": str(item.get("utterance", "")).strip(),
                    "source": "CLINC150"
                }
                for item in selected
            ]

        from datasets import load_dataset
        clinc_raw = load_dataset("DeepPavlov/clinc150", name="default", split="train")
        c_target = min(self.target_count, len(clinc_raw))
        clinc_sample = clinc_raw.shuffle(seed=self.config.random_seed).select(range(c_target))

        return [
            {
                "raw_text": str(row["utterance"]).strip(),
                "source": "CLINC150"
            }
            for row in clinc_sample
        ]

class Hwu64Adapter(DatasetAdapter):
    def __init__(self, config: PipelineConfig):
        super().__init__("HWU64", config.target_hwu64, config)

    def fetch_records(self) -> List[dict]:
        cache_file = self.ensure_source_file("hwu64.json") or os.path.join(self.config.cache_dir, "sources", "hwu64.json")
        if os.path.exists(cache_file):
            raw = json.load(open(cache_file, "r", encoding="utf-8"))
            h_target = min(self.target_count, len(raw))
            rng = random.Random(self.config.random_seed)
            shuffled = list(raw)
            rng.shuffle(shuffled)
            selected = shuffled[:h_target]
            return [
                {
                    "raw_text": str(item.get("utterance", "")).strip(),
                    "source": "HWU64"
                }
                for item in selected
            ]

        from datasets import load_dataset
        hwu_raw = load_dataset("DeepPavlov/hwu64", name="default", split="train")
        h_target = min(self.target_count, len(hwu_raw))
        hwu_sample = hwu_raw.shuffle(seed=self.config.random_seed).select(range(h_target))

        return [
            {
                "raw_text": str(row["utterance"]).strip(),
                "source": "HWU64"
            }
            for row in hwu_sample
        ]

class Minds14USAdapter(DatasetAdapter):
    def __init__(self, config: PipelineConfig):
        super().__init__("MINDS14_EN_US", config.target_minds14_us, config)

    def fetch_records(self) -> List[dict]:
        cache_file = self.ensure_source_file("minds14_us.json") or os.path.join(self.config.cache_dir, "sources", "minds14_us.json")
        if os.path.exists(cache_file):
            raw = json.load(open(cache_file, "r", encoding="utf-8"))
            m_target = min(self.target_count, len(raw))
            return [
                {
                    "raw_text": str(item.get("transcript", "")).strip(),
                    "source": "MINDS14_EN_US"
                }
                for item in raw[:m_target]
            ]

        from datasets import load_dataset
        minds_raw = load_dataset("PolyAI/minds14", name="en-US", split="train").remove_columns(["audio"])
        m_target = min(self.target_count, len(minds_raw))
        minds_sample = minds_raw.shuffle(seed=self.config.random_seed).select(range(m_target))

        return [
            {
                "raw_text": str(row.get("english_transcription") or row.get("transcription", "")).strip(),
                "source": "MINDS14_EN_US"
            }
            for row in minds_sample
        ]

class Minds14ExtAdapter(DatasetAdapter):
    def __init__(self, config: PipelineConfig):
        super().__init__("MINDS14_EXT", config.target_minds14_ext, config)

    def fetch_records(self) -> List[dict]:
        cache_file = self.ensure_source_file("minds14_ext.json") or os.path.join(self.config.cache_dir, "sources", "minds14_ext.json")
        if os.path.exists(cache_file):
            raw = json.load(open(cache_file, "r", encoding="utf-8"))
            records = []
            for item in raw:
                src = "MINDS14_EN_AU" if item.get("lang_id") == "en-AU" else "MINDS14_EN_GB"
                records.append({
                    "raw_text": str(item.get("transcript", "")).strip(),
                    "source": src
                })
            return records[:self.target_count]

        from datasets import load_dataset
        ds_au = load_dataset("PolyAI/minds14", name="en-AU", split="train").remove_columns(["audio"])
        ds_gb = load_dataset("PolyAI/minds14", name="en-GB", split="train").remove_columns(["audio"])

        target_au = min(654, len(ds_au))
        target_gb = min(592, len(ds_gb))

        sample_au = ds_au.shuffle(seed=self.config.random_seed).select(range(target_au))
        sample_gb = ds_gb.shuffle(seed=self.config.random_seed).select(range(target_gb))

        records = [
            {
                "raw_text": str(row.get("english_transcription") or row.get("transcription", "")).strip(),
                "source": "MINDS14_EN_AU"
            }
            for row in sample_au
        ]
        records.extend([
            {
                "raw_text": str(row.get("english_transcription") or row.get("transcription", "")).strip(),
                "source": "MINDS14_EN_GB"
            }
            for row in sample_gb
        ])
        return records

class CordV2Adapter(DatasetAdapter):
    def __init__(self, config: PipelineConfig):
        super().__init__("CORD-v2", config.target_cord_v2, config)

    def fetch_records(self) -> List[dict]:
        cache_file = self.ensure_source_file("cord_v2.json") or os.path.join(self.config.cache_dir, "sources", "cord_v2.json")
        if os.path.exists(cache_file):
            raw = json.load(open(cache_file, "r", encoding="utf-8"))
            c_target = min(self.target_count, len(raw))
            return [
                {
                    "raw_text": str(item.get("document_text", "")).strip(),
                    "source": "CORD-v2"
                }
                for item in raw[:c_target]
            ]

        from datasets import load_dataset
        ds = load_dataset("naver-clova-ix/cord-v2", split="train", streaming=True)
        records = []
        for item in ds:
            gt = item.get("ground_truth", "")
            gt_parse = {}
            if isinstance(gt, str):
                try: gt_parse = json.loads(gt).get("gt_parse", {})
                except Exception: gt_parse = {}
            elif isinstance(gt, dict) and "gt_parse" in gt:
                gt_parse = gt["gt_parse"]

            tokens = []
            def walk(o):
                if isinstance(o, str): tokens.append(o)
                elif isinstance(o, dict):
                    for v in o.values(): walk(v)
                elif isinstance(o, list):
                    for x in o: walk(x)
            walk(gt_parse)
            doc_text = " ".join(tokens) if tokens else "Receipt document"

            records.append({
                "raw_text": doc_text,
                "source": "CORD-v2"
            })
            if len(records) >= self.target_count:
                break
        return records

class SroieAdapter(DatasetAdapter):
    def __init__(self, config: PipelineConfig):
        super().__init__("SROIE", config.target_sroie, config)

    def fetch_records(self) -> List[dict]:
        cache_file = self.ensure_source_file("sroie.json") or os.path.join(self.config.cache_dir, "sources", "sroie.json")
        if os.path.exists(cache_file):
            raw = json.load(open(cache_file, "r", encoding="utf-8"))
            s_target = min(self.target_count, len(raw))
            return [
                {
                    "raw_text": str(item.get("document_text", "")).strip(),
                    "source": "SROIE"
                }
                for item in raw[:s_target]
            ]

        from datasets import load_dataset
        ds = load_dataset("rth/sroie-2019-v2", split="train")
        records = []
        for item in ds:
            objs = item.get("objects", {})
            texts = objs.get("text", objs.get("texts", objs.get("words", [])))
            doc_text = " ".join(str(t) for t in texts) if texts else "Receipt document"
            records.append({
                "raw_text": doc_text,
                "source": "SROIE"
            })
            if len(records) >= self.target_count:
                break
        return records

class FunsdAdapter(DatasetAdapter):
    def __init__(self, config: PipelineConfig):
        super().__init__("FUNSD", config.target_funsd, config)

    def fetch_records(self) -> List[dict]:
        cache_file = self.ensure_source_file("funsd.json") or os.path.join(self.config.cache_dir, "sources", "funsd.json")
        if os.path.exists(cache_file):
            raw = json.load(open(cache_file, "r", encoding="utf-8"))
            f_target = min(self.target_count, len(raw))
            return [
                {
                    "raw_text": str(item.get("document_text", "")).strip(),
                    "source": "FUNSD"
                }
                for item in raw[:f_target]
            ]

        from datasets import load_dataset
        ds = load_dataset("nielsr/funsd", split="train")
        records = []
        for item in ds:
            doc_text = " ".join(item.get("words", []))
            records.append({
                "raw_text": doc_text,
                "source": "FUNSD"
            })
            if len(records) >= self.target_count:
                break
        return records

print("All dataset adapters defined. Returning strictly raw_text + source.")


## 8. Original Dataset Builder
Builds the unified dataset containing ONLY `raw_text` and `source` (exactly 23,384 rows, 0 shortfall).

In [ ]:
# 8. Original Dataset Builder Execution
adapters = [
    Banking77Adapter(config),
    Clinc150Adapter(config),
    Hwu64Adapter(config),
    Minds14USAdapter(config),
    Minds14ExtAdapter(config),
    CordV2Adapter(config),
    SroieAdapter(config),
    FunsdAdapter(config)
]

all_original_records = []
adapter_stats = []

for ad in adapters:
    t0 = time.time()
    records = ad.fetch_records()
    available = len(records)

    # Filter only truly empty strings; keep real-world noise/mistakes
    valid_records = []
    excluded_count = 0
    for r in records:
        txt = str(r.get("raw_text", "")).strip()
        if len(txt) > 0:
            valid_records.append({"raw_text": txt, "source": r["source"]})
        else:
            excluded_count += 1

    target = min(ad.target_count, len(valid_records))
    selected = valid_records[:target]
    shortfall = max(0, ad.target_count - len(selected))

    all_original_records.extend(selected)
    stats = {
        "source": ad.source_name,
        "requested": ad.target_count,
        "available": available,
        "valid": len(valid_records),
        "excluded_empty": excluded_count,
        "selected": len(selected),
        "shortfall": shortfall
    }
    adapter_stats.append(stats)
    print(f"[{ad.source_name:14}] Avail: {available:,} | Sel: {len(selected):,} | Shortfall: {shortfall} ({time.time()-t0:.2f}s)")

df_original = pd.DataFrame(all_original_records)
print(f"
Total Unified Original Records: {len(df_original):,}")


## 9. Source Data Inspection
Verifies that the original dataset contains exactly two columns: `raw_text` and `source`.

In [ ]:
# 9. Verify Schema and Inspect Samples Per Source
assert list(df_original.columns) == ["raw_text", "source"], f"Columns violation: {list(df_original.columns)}"
assert df_original["raw_text"].notna().all(), "Null raw_text found!"
assert df_original["source"].notna().all(), "Null source found!"
assert (df_original["raw_text"].str.strip() != "").all(), "Empty raw_text found!"
assert len(df_original) == config.total_target, f"Expected {config.total_target} rows, got {len(df_original)}"

print("Columns in df_original strictly verified:", list(df_original.columns))
print(f"Total rows strictly verified: {len(df_original):,} (Target: {config.total_target:,})")

print("
--- 3 SAMPLES PER SOURCE ---")
for src in df_original["source"].unique():
    print(f"
[Source: {src}]")
    sub = df_original[df_original["source"] == src]
    samples = sub.sample(n=min(3, len(sub)), random_state=42)
    for i, (_, row) in enumerate(samples.iterrows(), 1):
        preview = row["raw_text"][:80].replace("
", " ")
        print(f"  {i}. {preview}...")


## 10. Sampling & Shortfall Audit
Audit table comparing requested targets vs selected records.

In [ ]:
# 10. Audit Table
audit_df = pd.DataFrame(adapter_stats)
print(audit_df[["source", "requested", "available", "selected", "shortfall"]].to_string(index=False))
print(f"
Total Selected: {audit_df['selected'].sum():,} / {audit_df['requested'].sum():,}")
print(f"Total Shortfall: {audit_df['shortfall'].sum():,}")


## 11. Original Dataset Export
Save `original_dataset.csv` with strictly `raw_text` and `source`.

In [ ]:
# 11. Save original_dataset.csv
os.makedirs(config.output_dir, exist_ok=True)
orig_csv_path = os.path.join(config.output_dir, "original_dataset.csv")
df_original.to_csv(orig_csv_path, index=False)
print(f"Saved {orig_csv_path} ({len(df_original):,} rows, {os.path.getsize(orig_csv_path):,} bytes)")
print("Columns:", list(df_original.columns))


## 12. LLM Provider Abstraction
OpenRouter client with strict quota and retry controls.

In [ ]:
# 12. OpenRouter Provider
class LLMProvider(ABC):
    @abstractmethod
    def generate(self, prompt: str, system_prompt: Optional[str] = None, is_test: bool = False) -> str:
        pass

class OpenRouterProvider(LLMProvider):
    def __init__(self, config: PipelineConfig, ledger: RequestLedger):
        self.config = config
        self.ledger = ledger
        self.api_key = os.environ.get("OPENROUTER_API_KEY", "")
        try:
            from google.colab import userdata
            self.api_key = self.api_key or userdata.get("OPENROUTER_API_KEY", "")
        except Exception:
            pass

    def is_configured(self) -> bool:
        return bool(self.api_key)

    def generate(self, prompt: str, system_prompt: Optional[str] = None, is_test: bool = False) -> str:
        if not self.is_configured():
            raise RuntimeError("OPENROUTER_API_KEY is not configured.")
        if not is_test and not self.ledger.can_request():
            raise RuntimeError(f"Production request limit reached ({self.config.daily_request_limit}/day).")

        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
            "HTTP-Referer": "https://latentspace.ai",
            "X-Title": "LatentSpace Dataset Pipeline"
        }
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": prompt})

        payload = {
            "model": self.config.annotation_model,
            "messages": messages,
            "response_format": {"type": "json_object"}
        }

        req = urllib.request.Request(
            f"{self.config.api_base_url}/chat/completions",
            data=json.dumps(payload).encode("utf-8"),
            headers=headers,
            method="POST"
        )

        t0 = time.time()
        attempt_id = f"req_{int(time.time()*1000)}"
        try:
            with urllib.request.urlopen(req, timeout=self.config.request_timeout) as resp:
                body = json.loads(resp.read().decode("utf-8"))
                latency = int((time.time() - t0) * 1000)
                self.ledger.record_attempt({
                    "timestamp": datetime.now(timezone.utc).isoformat(),
                    "attempt_id": attempt_id,
                    "is_test": is_test,
                    "model": self.config.annotation_model,
                    "http_status": resp.status,
                    "error_category": None,
                    "latency_ms": latency
                })
                return body["choices"][0]["message"]["content"]
        except Exception as e:
            latency = int((time.time() - t0) * 1000)
            self.ledger.record_attempt({
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "attempt_id": attempt_id,
                "is_test": is_test,
                "model": self.config.annotation_model,
                "http_status": 500,
                "error_category": type(e).__name__,
                "latency_ms": latency
            })
            raise e

provider = OpenRouterProvider(config, ledger)
print(f"Provider initialized. Configured: {provider.is_configured()}")


## 13. Pre-Flight Connectivity Test
Validates OpenRouter API access without consuming production quota.

In [ ]:
# 13. Pre-Flight Test
if provider.is_configured():
    try:
        resp = provider.generate("Respond with a test JSON object: {"status": "ok"}", is_test=True)
        print("Pre-flight connectivity test succeeded:", resp[:60])
    except Exception as e:
        print("Pre-flight test failed:", e)
else:
    print("Pre-flight test skipped: No API key configured. Strict True LLM mode active.")


## 14. Agentic Schema Completion & Synthesis Engine
Transforms raw_text into structured agentic actions using true LLM provider.
Strict True LLM Only: Never uses fake heuristic fallback. Failed annotations are isolated.

In [ ]:
# 14. Annotation Engine (Strict True LLM)
class AnnotationEngine:
    SYSTEM_PROMPT = """You are an expert agentic data annotation engine for the LatentSpace semantic middleware.
Your task is to analyze user raw input text and generate a structured agentic action plan adhering strictly to the provided taxonomy and universal tool contract: `dispatch_actions`.

Taxonomy Domains:
- finance (categories: finance/account, finance/transactions, finance/payments, finance/transfers, finance/cards, finance/expenses)
- account_and_security (categories: account_and_security/security, account_and_security/cards)
- travel (categories: travel/flights, travel/reservations, travel/transportation)
- productivity (categories: productivity/reminders, productivity/tasks, productivity/alarms, productivity/calendar)
- communication (categories: communication/messages, communication/email, communication/contacts)
- commerce (categories: commerce/orders)
- information (categories: information/general_information, information/navigation)
- documents (categories: documents/receipts, documents/forms)
- other (categories: other/clarification, other/out_of_scope)

Tool Contract:
If an action should be taken, emit tool_calls with function 'dispatch_actions' having argument {"actions": [{"action_id": "a1", "intent_name": "<registered_intent>", "depends_on": [], "entities": { ... }}]}.
If input is noisy, ambiguous, incomplete, or out of scope, do not invoke actions; set intents to [], tool_calls to [], and expected_response to a helpful clarification question or out-of-scope refusal.
Never fabricate execution results.

Output MUST be a single raw JSON object with keys:
{
  "domain": string,
  "category": string,
  "goal": string,
  "intents": list of action dicts,
  "entities": dict,
  "missing_information": list of strings,
  "requires_confirmation": boolean,
  "execution_plan": list of strings,
  "tool_calls": list of tool call dicts,
  "expected_response": string
}"""

    def __init__(self, config: PipelineConfig, registry: IntentRegistry, provider: Optional[LLMProvider] = None):
        self.config = config
        self.registry = registry
        self.provider = provider
        self.cache_dir = os.path.join(config.cache_dir, "annotations")
        os.makedirs(self.cache_dir, exist_ok=True)

    def annotate_record(self, record_id: str, raw_text: str, source: str) -> Tuple[Optional[dict], Optional[dict]]:
        cache_path = os.path.join(self.cache_dir, f"{record_id}.json")
        if os.path.exists(cache_path):
            cached = CheckpointManager.load_json(cache_path)
            if cached and cached.get("validation_status") == "accepted":
                return cached, None

        if self.provider is None or not self.provider.is_configured():
            return None, None

        user_content = f"Source: {source}
Raw Text: {raw_text}"

        try:
            resp_str = self.provider.generate(prompt=user_content, system_prompt=self.SYSTEM_PROMPT)
            clean_str = resp_str.strip()
            if clean_str.startswith("```json"):
                clean_str = clean_str[7:]
            if clean_str.startswith("```"):
                clean_str = clean_str[3:]
            if clean_str.endswith("```"):
                clean_str = clean_str[:-3]
            clean_str = clean_str.strip()

            parsed = json.loads(clean_str)

            annotated = {
                "record_id": record_id,
                "source": source,
                "raw_text": raw_text,
                "domain": str(parsed.get("domain", "other")),
                "category": str(parsed.get("category", "other/out_of_scope")),
                "goal": str(parsed.get("goal", "")),
                "intents": parsed.get("intents", []),
                "entities": parsed.get("entities", {}),
                "missing_information": parsed.get("missing_information", []),
                "requires_confirmation": bool(parsed.get("requires_confirmation", False)),
                "execution_plan": parsed.get("execution_plan", []),
                "tool_calls": parsed.get("tool_calls", []),
                "expected_response": str(parsed.get("expected_response", "")),
                "model": self.config.annotation_model,
                "validation_status": "pending"
            }

            return annotated, None

        except Exception as e:
            failed_record = {
                "record_id": record_id,
                "source": source,
                "raw_text": raw_text,
                "error_category": "llm_call_or_parse_error",
                "error_message": str(e),
                "raw_llm_response": "",
                "timestamp": datetime.now(timezone.utc).isoformat()
            }
            return None, failed_record

annotation_engine = AnnotationEngine(config, registry, provider)
print("AnnotationEngine initialized in Strict True LLM mode.")


## 15. Validation Engine
Deterministic structural validation against IntentRegistry and `dispatch_actions` universal tool contract.

In [ ]:
# 15. Validation Engine
class ValidationEngine:
    def __init__(self, registry: IntentRegistry):
        self.registry = registry

    def validate(self, annotated: dict) -> Tuple[bool, Optional[str], Optional[str]]:
        domain = annotated.get("domain")
        category = annotated.get("category")
        intents = annotated.get("intents", [])

        if domain not in DOMAINS:
            return False, "invalid_domain", f"Domain '{domain}' not in taxonomy."
        if category not in CATEGORIES:
            return False, "invalid_category", f"Category '{category}' not in taxonomy."

        action_ids = set()
        for act in intents:
            aid = act.get("action_id")
            iname = act.get("intent_name")
            if not aid:
                return False, "missing_action_id", "Missing action_id."
            if aid in action_ids:
                return False, "duplicate_action_id", f"Duplicate action_id '{aid}'."
            action_ids.add(aid)

            if not self.registry.is_valid_intent(iname):
                return False, "unregistered_intent", f"Intent '{iname}' not registered in registry."

            for dep in act.get("depends_on", []):
                if dep not in action_ids:
                    return False, "invalid_dependency", f"Dependency '{dep}' does not precede '{aid}'."

        tool_calls = annotated.get("tool_calls", [])
        for tc in tool_calls:
            if tc.get("function", {}).get("name") != "dispatch_actions":
                return False, "invalid_tool_name", "Tool name must strictly be 'dispatch_actions'."
            try:
                args = json.loads(tc.get("function", {}).get("arguments", "{}"))
                if "actions" not in args or not isinstance(args["actions"], list):
                    return False, "malformed_tool_arguments", "Tool arguments missing 'actions' list."
            except Exception as e:
                return False, "malformed_json_arguments", f"Tool arguments JSON parse error: {e}"

        return True, None, None

validation_engine = ValidationEngine(registry)
print("ValidationEngine initialized.")


## 16. Synthetic Dataset Extraction
Extracts only the verified LLM-annotated records. Unannotated or failed records are cleanly separated.

In [ ]:
# 16. Synthetic Dataset Extraction Execution
annotated_records = []
failed_records = []

if provider.is_configured():
    limit = config.daily_request_limit
    print(f"Annotating records with True LLM ({config.annotation_model}, limit: {limit})...")
    for idx, row in df_original.iterrows():
        if len(annotated_records) >= limit:
            print(f"Reached limit of {limit} true LLM annotations.")
            break

        record_id = f"{row['source']}_{idx:06d}"
        raw_text = row["raw_text"]
        source = row["source"]

        ann, fail = annotation_engine.annotate_record(record_id, raw_text, source)
        if fail:
            failed_records.append(fail)
            continue
        if ann:
            is_valid, err_cat, err_msg = validation_engine.validate(ann)
            if is_valid:
                ann["validation_status"] = "accepted"
                annotated_records.append(ann)
            else:
                failed_records.append({
                    "record_id": record_id,
                    "source": source,
                    "raw_text": raw_text,
                    "error_category": err_cat or "validation_failure",
                    "error_message": err_msg or "Failed schema validation",
                    "raw_llm_response": json.dumps(ann, ensure_ascii=False),
                    "timestamp": datetime.now(timezone.utc).isoformat()
                })
else:
    print("Strict True LLM Mode: No API key configured. 0 synthetic records generated.")
    print("All 23,384 records remain cleanly in original_dataset.csv as pending.")

print(f"Verified True LLM Records: {len(annotated_records):,}")
print(f"Failed Annotations Isolated: {len(failed_records):,}")


## 17. Deduplication & Integrity Check
Checks for semantic and lexical integrity across annotated records.

In [ ]:
# 17. Deduplication & Integrity Audit
rec_ids = [r["record_id"] for r in annotated_records]
print(f"Total annotated records: {len(annotated_records):,}")
print(f"Unique record IDs: {len(set(rec_ids)):,}")
assert len(rec_ids) == len(set(rec_ids)), "Duplicate record IDs found in annotated records!"
print("Integrity check passed.")


## 18. Train / Validation / Test Split (Zero Leakage)
Executes group-aware 80% Train, 10% Validation, 10% Test split based on raw_text to ensure absolute zero text leakage.

In [ ]:
# 18. Zero-Leakage Split Manager
class SplitManager:
    @staticmethod
    def assign_splits(records: List[dict], seed: int = 42) -> Tuple[List[dict], List[dict], List[dict]]:
        if not records:
            return [], [], []

        rng = random.Random(seed)

        groups: Dict[str, List[dict]] = {}
        for r in records:
            key = r.get("raw_text", "").strip().lower()
            groups.setdefault(key, []).append(r)

        unique_keys = list(groups.keys())
        rng.shuffle(unique_keys)

        total_records = len(records)
        target_train = int(round(0.80 * total_records))
        target_val = int(round(0.10 * total_records))

        train, val, test = [], [], []

        for key in unique_keys:
            group = groups[key]
            if len(train) + len(group) <= target_train or (len(val) >= target_val and len(test) >= (total_records - target_train - target_val)):
                train.extend(group)
            elif len(val) + len(group) <= target_val:
                val.extend(group)
            else:
                test.extend(group)

        for r in train: r["split"] = "train"
        for r in val: r["split"] = "validation"
        for r in test: r["split"] = "test"

        return train, val, test

    @staticmethod
    def verify_no_leakage(train: List[dict], val: List[dict], test: List[dict]) -> bool:
        if not train and not val and not test:
            return True

        s_train = {r["record_id"] for r in train}
        s_val = {r["record_id"] for r in val}
        s_test = {r["record_id"] for r in test}

        assert len(s_train.intersection(s_val)) == 0, "Leakage between train and validation!"
        assert len(s_train.intersection(s_test)) == 0, "Leakage between train and test!"
        assert len(s_val.intersection(s_test)) == 0, "Leakage between validation and test!"

        t_train = {r.get("raw_text", "").strip().lower() for r in train}
        t_val = {r.get("raw_text", "").strip().lower() for r in val}
        t_test = {r.get("raw_text", "").strip().lower() for r in test}
        assert len(t_train.intersection(t_val)) == 0, "Text leakage between train and validation!"
        assert len(t_train.intersection(t_test)) == 0, "Text leakage between train and test!"
        assert len(t_val.intersection(t_test)) == 0, "Text leakage between validation and test!"
        return True

train_records, val_records, test_records = SplitManager.assign_splits(annotated_records, seed=config.random_seed)
SplitManager.verify_no_leakage(train_records, val_records, test_records)
print(f"Splits generated: Train = {len(train_records):,} | Val = {len(val_records):,} | Test = {len(test_records):,}")
print("Zero text and record leakage verified across splits.")


## 19. Final JSONL Export
Exports OpenAI-style conversation JSONL files with canonical `dispatch_actions` tool call, and exports failed annotations to a separate file.

In [ ]:
# 19. Final JSONL Export & Failed Annotations Isolation
class DatasetExporter:
    SYSTEM_MESSAGE = "You are a helpful, autonomous AI assistant. You help users achieve their goals by utilizing the tools provided to you."

    @staticmethod
    def format_jsonl_record(record: dict) -> dict:
        raw_text = record.get("raw_text", "")
        tool_calls = record.get("tool_calls", [])
        expected_response = record.get("expected_response", "")

        messages = [
            {"role": "system", "content": DatasetExporter.SYSTEM_MESSAGE},
            {"role": "user", "content": raw_text}
        ]

        if tool_calls:
            messages.append({
                "role": "assistant",
                "content": "",
                "tool_calls": tool_calls
            })
        else:
            messages.append({
                "role": "assistant",
                "content": expected_response
            })

        return {
            "messages": messages,
            "tools": [UNIVERSAL_DISPATCH_TOOL]
        }

    @staticmethod
    def export_all(
        config: PipelineConfig,
        annotated_records: List[dict],
        failed_records: List[dict],
        train_records: List[dict],
        val_records: List[dict],
        test_records: List[dict]
    ):
        os.makedirs(config.output_dir, exist_ok=True)

        # 1. output/failed_annotations.csv and .jsonl (Dedicated failure file)
        failed_columns = ["record_id", "source", "raw_text", "error_category", "error_message", "raw_llm_response", "timestamp"]
        df_failed = pd.DataFrame(failed_records, columns=failed_columns) if failed_records else pd.DataFrame(columns=failed_columns)
        failed_csv_path = os.path.join(config.output_dir, "failed_annotations.csv")
        df_failed.to_csv(failed_csv_path, index=False)
        failed_jsonl_path = os.path.join(config.output_dir, "failed_annotations.jsonl")
        with open(failed_jsonl_path, "w", encoding="utf-8") as f:
            for r in failed_records:
                f.write(json.dumps(r, ensure_ascii=False) + "
")
        print(f"Exported failed_annotations.csv ({len(df_failed):,} rows)")
        print(f"Exported failed_annotations.jsonl ({len(df_failed):,} rows)")

        # 2. output/synthetic_dataset.csv
        synth_columns = [
            "record_id", "source", "raw_text", "domain", "category", "goal",
            "intents", "entities", "missing_information", "requires_confirmation",
            "execution_plan", "tool_calls", "expected_response", "model"
        ]
        synthetic_records = [
            {
                "record_id": r["record_id"],
                "source": r["source"],
                "raw_text": r["raw_text"],
                "domain": r["domain"],
                "category": r["category"],
                "goal": r["goal"],
                "intents": json.dumps(r["intents"]),
                "entities": json.dumps(r["entities"]),
                "missing_information": json.dumps(r["missing_information"]),
                "requires_confirmation": r["requires_confirmation"],
                "execution_plan": json.dumps(r["execution_plan"]),
                "tool_calls": json.dumps(r["tool_calls"]),
                "expected_response": r["expected_response"],
                "model": r.get("model", config.annotation_model)
            }
            for r in annotated_records
        ]
        df_synth = pd.DataFrame(synthetic_records, columns=synth_columns) if synthetic_records else pd.DataFrame(columns=synth_columns)
        synth_path = os.path.join(config.output_dir, "synthetic_dataset.csv")
        df_synth.to_csv(synth_path, index=False)
        print(f"Exported synthetic_dataset.csv ({len(df_synth):,} rows)")

        # 3. output/combined_dataset.csv
        comb_columns = ["record_id", "source", "raw_text", "domain", "category", "goal", "intents", "validation_status", "split"]
        df_comb = pd.DataFrame([
            {
                "record_id": r["record_id"],
                "source": r["source"],
                "raw_text": r["raw_text"],
                "domain": r["domain"],
                "category": r["category"],
                "goal": r["goal"],
                "intents": json.dumps(r["intents"]),
                "validation_status": r["validation_status"],
                "split": r.get("split", "")
            }
            for r in annotated_records
        ], columns=comb_columns) if annotated_records else pd.DataFrame(columns=comb_columns)
        comb_path = os.path.join(config.output_dir, "combined_dataset.csv")
        df_comb.to_csv(comb_path, index=False)
        print(f"Exported combined_dataset.csv ({len(df_comb):,} rows)")

        # 4. JSONL files
        for name, recs in [
            ("latentspace_complete_dataset.jsonl", annotated_records),
            ("latentspace_train.jsonl", train_records),
            ("latentspace_validation.jsonl", val_records),
            ("latentspace_test.jsonl", test_records)
        ]:
            out_file = os.path.join(config.output_dir, name)
            with open(out_file, "w", encoding="utf-8") as f:
                for r in recs:
                    formatted = DatasetExporter.format_jsonl_record(r)
                    f.write(json.dumps(formatted, ensure_ascii=False) + "
")
            print(f"Exported {name} ({len(recs):,} records)")

DatasetExporter.export_all(
    config,
    annotated_records,
    failed_records,
    train_records,
    val_records,
    test_records
)


## 20. Reporting
Generates `output/dataset_report.json` compiling all distribution metrics, audit stats, and LLM usage ledger.

In [ ]:
# 20. Quality Report Generation
class QualityReporter:
    @staticmethod
    def generate_report(
        config: PipelineConfig,
        adapter_stats: List[dict],
        total_original_count: int,
        annotated_records: List[dict],
        failed_records: List[dict],
        train_records: List[dict],
        val_records: List[dict],
        test_records: List[dict],
        ledger: RequestLedger
    ) -> dict:
        domain_counts, category_counts, intent_counts, source_counts = {}, {}, {}, {}
        for r in annotated_records:
            d = r.get("domain", "other")
            c = r.get("category", "other/out_of_scope")
            s = r.get("source", "unknown")
            domain_counts[d] = domain_counts.get(d, 0) + 1
            category_counts[c] = category_counts.get(c, 0) + 1
            source_counts[s] = source_counts.get(s, 0) + 1
            for act in r.get("intents", []):
                iname = act.get("intent_name", "unknown")
                intent_counts[iname] = intent_counts.get(iname, 0) + 1

        pending_count = total_original_count - len(annotated_records) - len(failed_records)

        report = {
            "pipeline_version": config.pipeline_version,
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "random_seed": config.random_seed,
            "total_original_records": total_original_count,
            "total_requested": sum(s["requested"] for s in adapter_stats),
            "total_selected": sum(s["selected"] for s in adapter_stats),
            "total_shortfall": sum(s["shortfall"] for s in adapter_stats),
            "source_breakdown": {s["source"]: {"requested": s["requested"], "selected": s["selected"], "shortfall": s["shortfall"]} for s in adapter_stats},
            "annotation_metrics": {
                "true_llm_annotated_count": len(annotated_records),
                "failed_annotation_count": len(failed_records),
                "pending_unannotated_count": pending_count,
                "zero_fake_annotations_verified": True
            },
            "train_count": len(train_records),
            "validation_count": len(val_records),
            "test_count": len(test_records),
            "domain_distribution": domain_counts,
            "category_distribution": category_counts,
            "intent_distribution": intent_counts,
            "production_requests_used": ledger.production_requests_used,
            "cache_hits": ledger.cache_hits,
            "retry_count": ledger.retries
        }

        report_path = os.path.join(config.output_dir, "dataset_report.json")
        CheckpointManager.atomic_write_json(report_path, report)
        print("Exported dataset_report.json successfully.")
        return report

report = QualityReporter.generate_report(
    config,
    adapter_stats,
    len(df_original),
    annotated_records,
    failed_records,
    train_records,
    val_records,
    test_records,
    ledger
)


## 21. Final Verification
Comprehensive programmatic verification across all exported files, schemas, targets, and leakage assertions.

In [ ]:
# 21. Programmatic Verification & Colab Download Helper
print("=" * 80)
print("RUNNING FINAL PROGRAMMATIC VERIFICATION")
print("=" * 80)

# 1. Verify original_dataset.csv
orig_csv = os.path.join(config.output_dir, "original_dataset.csv")
assert os.path.exists(orig_csv), "original_dataset.csv missing!"
df_orig_chk = pd.read_csv(orig_csv)
assert list(df_orig_chk.columns) == ["raw_text", "source"], f"Wrong columns: {list(df_orig_chk.columns)}"
assert len(df_orig_chk) == config.total_target, f"Expected {config.total_target} rows, got {len(df_orig_chk)}"
assert df_orig_chk["raw_text"].notna().all(), "Null raw_text found!"
assert df_orig_chk["source"].notna().all(), "Null source found!"
assert (df_orig_chk["raw_text"].str.strip() != "").all(), "Empty raw_text found!"
print(f"[PASSED] original_dataset.csv: strictly 2 columns, exactly {len(df_orig_chk):,} rows, 0 shortfall.")

# 2. Verify failed_annotations.csv
failed_csv = os.path.join(config.output_dir, "failed_annotations.csv")
assert os.path.exists(failed_csv), "failed_annotations.csv missing!"
df_failed_chk = pd.read_csv(failed_csv)
print(f"[PASSED] failed_annotations.csv: verified ({len(df_failed_chk):,} failed records isolated).")

# 3. Verify synthetic_dataset.csv
synth_csv = os.path.join(config.output_dir, "synthetic_dataset.csv")
assert os.path.exists(synth_csv), "synthetic_dataset.csv missing!"
df_synth_chk = pd.read_csv(synth_csv)
print(f"[PASSED] synthetic_dataset.csv: verified ({len(df_synth_chk):,} true LLM records).")

# 4. Verify JSONL files
for split_name in ["latentspace_complete_dataset.jsonl", "latentspace_train.jsonl", "latentspace_validation.jsonl", "latentspace_test.jsonl"]:
    fpath = os.path.join(config.output_dir, split_name)
    assert os.path.exists(fpath), f"{split_name} missing!"
    line_count = sum(1 for _ in open(fpath, "r", encoding="utf-8"))
    print(f"[PASSED] {split_name}: {line_count:,} records.")

# 5. Verify dataset_report.json
report_path = os.path.join(config.output_dir, "dataset_report.json")
assert os.path.exists(report_path), "dataset_report.json missing!"
print("[PASSED] dataset_report.json verified.")

print("
" + "=" * 80)
print("ALL VERIFICATION CHECKS PASSED SUCCESSFULLY!")
print("=" * 80)

# Optional download helper for Google Colab
try:
    from google.colab import files
    print("
To download outputs in Google Colab, uncomment:")
    print("# files.download('latentspace_dataset/output/original_dataset.csv')")
except ImportError:
    pass
